In [8]:
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("API_KEY")

In [9]:
from google import genai

from google.genai import types

client = genai.Client(api_key=api_key)
model = "gemini-2.5-flash"

In [10]:
def add_user_message(messages, content):
    user_message = {"role": "user", "parts": [{"text": content}]}
    messages.append(user_message)


def add_assistant_message(messages, content):
    assistant_message = {"role": "model", "parts": [{"text": content}]}
    messages.append(assistant_message)


def chat(messages, stop_sequences = None):
    response = client.models.generate_content(
        model=model,
        contents=messages,
        config=types.GenerateContentConfig(
            stop_sequences=stop_sequences
        )
    )
    return response

In [5]:
messages = []
while True:
    user_input = input(" > ")

    print("User input is:", user_input)

    if user_input.lower() == "exit":
        break

    add_user_message(messages, user_input)

    answer = chat(messages)

    add_assistant_message(messages, answer.text)

    print("----")
    print(answer.text)
    print("----")

User input is: What is 2 + 2 ? 
----
2 + 2 = **4**
----
User input is: What is quantum computing ? 
----
Quantum computing is a revolutionary new type of computing that harnesses the principles of **quantum mechanics** to perform calculations. Unlike classical computers that store information as bits (either a 0 or a 1), quantum computers use **qubits**.

Here's a breakdown of what that means:

### Key Concepts:

1.  **Qubits (Quantum Bits):**
    *   A classical bit can be only 0 or 1.
    *   A qubit can be 0, 1, or a **superposition** of both 0 and 1 simultaneously. Imagine a spinning coin that's neither heads nor tails until it lands. This allows a single qubit to hold more information than a classical bit.

2.  **Superposition:**
    *   This is the ability of a qubit to exist in multiple states at once. If you have 'n' qubits in superposition, they can represent 2^n states simultaneously. This is where the massive parallel processing power comes from.

3.  **Entanglement:**
    *

In [15]:
import json
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    ans = chat(messages, stop_sequences=["```"])
    return json.loads(ans.text)


In [17]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent= 2)


In [42]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output.text

In [47]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement  
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_response = chat(messages, stop_sequences=["```"])
    return json.loads(eval_response.text)

In [52]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

In [48]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    
    model_grade = grade_by_model(test_case , output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning" : reasoning
    }

In [49]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [50]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [51]:
print(results)


[{'output': 'The task is to extract the AWS Region from an ARN string. An AWS ARN (Amazon Resource Name) follows a standard format:\n\n`arn:partition:service:region:account-id:resource`\n\nThe region component is the fourth part of the ARN when split by the colon (`:`) character (index 3 in a 0-indexed list). Some ARNs, particularly for global services like IAM or certain S3 bucket ARNs, do not explicitly include a region in the ARN string, in which case the region component will be an empty string.\n\nThe solution will:\n1. Split the ARN string by the colon delimiter.\n2. Check if there are at least 4 parts in the split string. If not, the string is malformed or too short to contain a region component, and an empty string should be returned.\n3. If there are enough parts, return the element at index 3. This will be the region if present, or an empty string if the region is not explicitly specified in the ARN.\n\n```python\ndef extract_aws_region_from_arn(arn_string: str) -> str:\n    